# RetentionAI — 01. Data Extraction

Downloads the Telco Customer Churn dataset from Kaggle, extracts it, and saves a clean copy to `data/raw/telco_churn.csv`.

Run this once. Everything downstream reads from that path.

In [2]:
import zipfile
import subprocess
from pathlib import Path
import pandas as pd

In [3]:
# Anchor to project root regardless of where the kernel starts.
# Notebooks live one level below root, so we go up if needed.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = RAW_DATA_DIR / "telco-customer-churn.zip"
CSV_PATH = RAW_DATA_DIR / "telco_churn.csv"

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw data dir : {RAW_DATA_DIR}")

Project root : d:\Important\00-Projects\RetentionAI
Raw data dir : d:\Important\00-Projects\RetentionAI\data\raw


### Download

Requires the Kaggle CLI and `~/.kaggle/kaggle.json`. See [Kaggle API setup](https://github.com/Kaggle/kaggle-api#api-credentials) if you haven't done this.

Skips the download if the CSV already exists — safe to re-run.

In [4]:
# Delete data/raw/telco_churn.csv to force a fresh download from Kaggle.
if CSV_PATH.exists():
    print("CSV already present, skipping download.")
else:
    print("Downloading from Kaggle...")
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", "blastchar/telco-customer-churn",
            "-p", str(RAW_DATA_DIR),
        ],
        check=True,   # raises CalledProcessError on non-zero exit
    )
    print("Done.")

Done.


### Extract and rename

Kaggle packages it as a ZIP. We extract, rename to a stable filename, and delete the ZIP — no reason to keep it around.

In [5]:
if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DATA_DIR)
    ZIP_PATH.unlink()  # delete ZIP after extracting
    print("Extracted and deleted ZIP.")

# Kaggle names the file 'WA_Fn-UseC_-Telco-Customer-Churn.csv'.
# Rename it once so every notebook downstream uses the same stable path.
kaggle_name = RAW_DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
if kaggle_name.exists() and not CSV_PATH.exists():
    kaggle_name.rename(CSV_PATH)
    print(f"Renamed to: {CSV_PATH.name}")

Extracted and deleted ZIP.
Renamed to: telco_churn.csv


### Verify

Quick sanity check — shape and a peek at the data. The assert will catch a corrupted download or a wrong file before anything downstream breaks silently.

In [6]:
df = pd.read_csv(CSV_PATH)

assert df.shape == (7043, 21), f"Unexpected shape: {df.shape}"

print(f"Shape: {df.shape}  — looks good.")
df.head()

Shape: (7043, 21)  — looks good.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
